# Pencarian Solusi Rotatable Edge-Matching Puzzle dengan Local Search

Notebook PoC untuk complete-state formulation, objective, neighbor, seluruh varian Hill-Climbing, Simulated Annealing, Genetic Algorithm, eksperimen, dan visualisasi.

**Global optimum 5x5 = 40 matched adjacency.**


## 1. Setup dan Instance

Setiap state berisi permutation seluruh tile dan rotasi setiap tile. Initial state dibangkitkan random.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

from src.local_search import EdgeMatchingProblem
from src.local_search.hill_climbing import (
    steepest_ascent, sideways_move,
    stochastic_hill_climbing, random_restart,
)
from src.local_search.simulated_annealing import simulated_annealing
from src.local_search.genetic_algorithm import genetic_algorithm
from src.local_search.visualization import (
    plot_board, plot_score_history,
    plot_simulated_annealing, plot_ga_history,
)

INSTANCE = ROOT / "src/local_search/instances/edge_matching_5x5.json"
problem = EdgeMatchingProblem.from_json(INSTANCE)
initial = problem.random_state(seed=42)
problem.score(initial), problem.max_score


In [ ]:
plot_board(problem, initial, title=f"Initial State — {problem.score(initial)}/{problem.max_score}")


## 2. Objective dan Neighbor

$$
f(S)=\sum_{(i,j)\in A}\mathbf{1}[edge_i=edge_j]
$$

Neighbor terdiri dari **Swap Move** dan **Rotate Move**. Untuk board 5x5, full neighborhood terdiri dari 300 swap + 75 rotation = 375 candidate state.


In [ ]:
neighbors = list(problem.iter_neighbors(initial))
len(neighbors), problem.score(neighbors[0].state)


## 3. Hill-Climbing


In [ ]:
hc = steepest_ascent(problem, initial, max_iterations=500, random_state=42)
hc.best_score, hc.iterations, hc.metadata["stop_reason"]


In [ ]:
plot_board(problem, hc.best_state, title=f"Steepest-Ascent — {hc.best_score}/40")
plot_score_history(hc)


In [ ]:
sideways = sideways_move(problem, initial, max_sideways=50, random_state=42)
stochastic = stochastic_hill_climbing(problem, initial, random_state=42)
restart = random_restart(problem, max_restarts=10, random_state=42)

[(x.algorithm, x.best_score, x.iterations) for x in [sideways, stochastic, restart]]


## 4. Simulated Annealing

Worse move diterima dengan probabilitas

$$
P(accept)=e^{\Delta f/T},\quad \Delta f<0.
$$


In [ ]:
sa = simulated_annealing(
    problem,
    initial,
    initial_temperature=8.0,
    cooling_rate=0.997,
    max_iterations=20000,
    random_state=42,
)
sa.best_score, sa.metadata["accepted_worse_moves"], sa.metadata["local_optimum_escape_events"]


In [ ]:
plot_board(problem, sa.best_state, title=f"Simulated Annealing — {sa.best_score}/40")
plot_simulated_annealing(sa)


## 5. Genetic Algorithm

Chromosome adalah permutation `(tile_id, rotation)`. Selection menggunakan tournament selection, crossover menggunakan **Order Crossover (OX)** agar setiap tile tetap unik, dan mutation terdiri dari swap/rotation mutation.


In [ ]:
ga = genetic_algorithm(
    problem,
    population_size=120,
    generations=600,
    random_state=42,
)
ga.best_score, ga.iterations


In [ ]:
plot_board(problem, ga.best_state, title=f"Genetic Algorithm — {ga.best_score}/40")
plot_ga_history(ga)


## 6. Eksperimen Spesifikasi

`run_standard_experiments()` menjalankan seluruh algoritma sebanyak 3 kali dan menyimpan state awal/akhir, trace, plot objective, serta durasi. `run_ga_parameter_sweep()` menjalankan skema variasi population size dan generations.


In [ ]:
from src.local_search.experiments import run_standard_experiments, run_ga_parameter_sweep

# Jalankan saat eksperimen final:
# summary = run_standard_experiments(problem, runs=3)
# ga_sweep = run_ga_parameter_sweep(problem, runs=3)


## 7. Bonus Replay

Hasil `SearchResult.save_json()` dapat dibuka menggunakan `ReplayPlayer` untuk play/pause, previous/next, progress slider, dan playback speed.


In [ ]:
from src.local_search.replay import ReplayPlayer

# ReplayPlayer(problem, sa).show()
